# 03 — Persiapan Data: Prapemrosesan Teks & Pelabelan

Tahapan prapemrosesan teks ulasan **Access by KAI** sebelum pemodelan:

1. **Cleaning** — buang URL, tag HTML, mention, hashtag, angka, tanda baca, emoji.
2. **Case folding** — ubah ke huruf kecil.
3. **Normalisasi slang** — samakan kata tidak baku (mis. `gak` -> `tidak`, `bagus bgt` -> `bagus sangat`).
4. **Stopword removal** — buang kata umum (Sastrawi), **negasi dipertahankan** (`tidak`, `bukan`, `jangan`, `belum`, `kurang`, `tanpa`).
5. **Stemming** — Sastrawi.
6. **Pelabelan** — bintang 4-5 = `positif` (1), bintang 1-2 = `negatif` (0); bintang 3 (netral) **dikeluarkan** dari klasifikasi biner.
7. **Train/test split** — stratified 80:20, `random_state=42`.

Input : `data/raw/reviews_access_by_kai.csv`  
Output: `data/processed/dataset_bersih.csv`, `data/processed/train.csv`, `data/processed/test.csv`

> TF-IDF **tidak** dihitung di sini; harus di-*fit* hanya pada data latih agar tidak terjadi *data leakage* (dilakukan di `04_model_klasifikasi.ipynb`).

In [1]:
from pathlib import Path
import re
import time

import pandas as pd
from sklearn.model_selection import train_test_split
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW / "reviews_access_by_kai.csv")
print("Baris mentah:", len(df))

def label_sentimen(score):
    if score >= 4:
        return "positif"
    if score <= 2:
        return "negatif"
    return "netral"

df["sentimen"] = df["score"].map(label_sentimen)
print(df["sentimen"].value_counts())

sebelum = len(df)
df = df[df["sentimen"] != "netral"].copy()
print(f"\nUlasan netral dibuang: {sebelum - len(df)}")
print("Data klasifikasi biner:", len(df))
print(df["sentimen"].value_counts())

Baris mentah: 5000
sentimen
positif    2706
negatif    1975
netral      319
Name: count, dtype: int64

Ulasan netral dibuang: 319
Data klasifikasi biner: 4681
sentimen
positif    2706
negatif    1975
Name: count, dtype: int64


In [2]:
slang_dict = {
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "g": "tidak", "tdk": "tidak",
    "engga": "tidak", "enggak": "tidak", "nggak": "tidak", "ngga": "tidak",
    "gada": "tidak ada", "gabisa": "tidak bisa", "gbs": "tidak bisa",
    "gaboleh": "tidak boleh", "gabisa": "tidak bisa",
    "udah": "sudah", "udh": "sudah", "uda": "sudah", "dah": "sudah", "sdh": "sudah",
    "banget": "sangat", "bgt": "sangat", "bngt": "sangat", "sngat": "sangat", "sngad": "sangat",
    "aja": "saja", "ajah": "saja", "doang": "saja",
    "yg": "yang", "dgn": "dengan", "utk": "untuk", "krn": "karena", "krna": "karena",
    "karna": "karena", "dri": "dari", "dr": "dari", "jd": "jadi", "jdi": "jadi",
    "tp": "tapi", "tpi": "tapi", "kalo": "kalau", "klo": "kalau", "kl": "kalau",
    "gimana": "bagaimana", "gmn": "bagaimana", "knp": "kenapa", "napa": "kenapa",
    "bener": "benar", "bner": "benar", "bnr": "benar",
    "mantappp": "mantap", "mantul": "mantap", "mantaap": "mantap",
    "bgus": "bagus", "bgs": "bagus", "jlk": "jelek",
    "susah": "sulit", "gampang": "mudah", "gmpg": "mudah",
    "hp": "ponsel", "wa": "whatsapp", "apk": "aplikasi", "app": "aplikasi",
    "aplikasinya": "aplikasi", "tiketnya": "tiket",
    "trims": "terima kasih", "makasih": "terima kasih", "mksh": "terima kasih",
    "thx": "terima kasih", "tks": "terima kasih", "thanks": "terima kasih", "thank": "terima kasih",
    "eror": "error", "erorr": "error", "err": "error",
    "lemot": "lambat", "lelet": "lambat",
    "cs": "customer service", "csnya": "customer service",
    "bkn": "bukan", "bukanya": "bukan",
    "kayak": "seperti", "kaya": "seperti", "kyk": "seperti",
    "jg": "juga", "sm": "sama", "sma": "sama", "bs": "bisa", "bsa": "bisa",
    "pgn": "ingin", "pengen": "ingin", "pengin": "ingin",
    "dapet": "dapat", "dpt": "dapat", "dapetin": "dapatkan",
    "bli": "beli", "byr": "bayar", "duit": "uang", "duid": "uang",
    "ilang": "hilang", "ribet": "rumit", "cpt": "cepat", "cepet": "cepat",
    "telad": "terlambat", "pake": "pakai", "pke": "pakai",
    "skrg": "sekarang", "skr": "sekarang", "kmrn": "kemarin", "kmrin": "kemarin",
    "bsk": "besok", "thn": "tahun", "taun": "tahun", "org": "orang",
    "tau": "tahu", "liat": "lihat", "liad": "lihat",
    "download": "unduh", "donlot": "unduh", "dwnload": "unduh",
    "install": "instal", "veritifikasi": "verifikasi",
    "pdhl": "padahal", "smoga": "semoga", "smg": "semoga", "mudah2an": "semoga",
}

NEGASI = {"tidak", "bukan", "jangan", "belum", "kurang", "tanpa"}
stopword = set(StopWordRemoverFactory().get_stop_words()) - NEGASI
stemmer = StemmerFactory().create_stemmer()
print("Jumlah stopword:", len(stopword))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_text(text):
    return " ".join(slang_dict.get(tok, tok) for tok in text.split())

_stem_cache = {}
def stem_word(word):
    if word not in _stem_cache:
        _stem_cache[word] = stemmer.stem(word)
    return _stem_cache[word]

def preprocess(text):
    teks = clean_text(text)
    teks = normalize_text(teks)
    tokens = [w for w in teks.split() if len(w) >= 3 and w not in stopword]
    tokens = [stem_word(w) for w in tokens]
    tokens = [w for w in tokens if w and w not in stopword]
    return " ".join(tokens)

Jumlah stopword: 120


In [3]:
contoh = df.loc[df["sentimen"] == "negatif", "content"].head(3).tolist()
print("=== Contoh sebelum -> sesudah ===")
for t in contoh:
    print("ASLI   :", t[:150])
    print("BERSIH :", preprocess(t))
    print()

t0 = time.time()
df["text_bersih"] = df["content"].map(preprocess)
print(f"Waktu prapemrosesan: {time.time() - t0:.1f} detik")

=== Contoh sebelum -> sesudah ===
ASLI   : kalo ga mau duit lu ilang mending jangan download apk ini mana banyak bug nya!
BERSIH : kalau tidak mau uang hilang mending jangan unduh aplikasi mana banyak bug nya

ASLI   : ketika cancel tiket veritifikasi kode otp sudah benar k tapi selalu tidak bisa. ini sengaja gak bs dicancel atau emang aplikasinya yang gak jelas. dic


BERSIH : cancel tiket verifikasi kode otp benar selalu tidak sengaja tidak dicancel emang aplikasi tidak jelas coba waktu sama

ASLI   : terlalu sering up -grade2,capek,aktivisasi kaipai butuh waktu,perasaan saldo kaipai sy msh 50k,hangus deh


BERSIH : terlalu sering grade capek aktivisasi kaipai butuh waktu asa saldo kaipai msh hangus deh



Waktu prapemrosesan: 252.2 detik


In [4]:
kosong = (df["text_bersih"].str.strip() == "").sum()
print("Teks kosong setelah preprocessing:", kosong)
df = df[df["text_bersih"].str.strip() != ""].copy()
print("Baris siap pakai:", len(df))

df["label"] = (df["sentimen"] == "positif").astype(int)
print("\nDistribusi label (0=negatif, 1=positif):")
print(df["label"].value_counts().sort_index())

df["n_kata_bersih"] = df["text_bersih"].str.split().map(len)
print("\nPanjang ulasan bersih (kata):")
print(df["n_kata_bersih"].describe())

Teks kosong setelah preprocessing: 163
Baris siap pakai: 4518

Distribusi label (0=negatif, 1=positif):
label
0    1973
1    2545
Name: count, dtype: int64

Panjang ulasan bersih (kata):
count    4518.000000
mean        8.914121
std        10.485119
min         1.000000
25%         2.000000
50%         5.000000
75%        12.000000
max        70.000000
Name: n_kata_bersih, dtype: float64


In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"],
)
print("Latih:", train_df.shape, "| Uji:", test_df.shape)
print("\nProporsi label latih:")
print(train_df["label"].value_counts(normalize=True).round(4))
print("Proporsi label uji:")
print(test_df["label"].value_counts(normalize=True).round(4))

kolom = ["reviewId", "content", "text_bersih", "score", "sentimen", "label"]
df[kolom].to_csv(DATA_PROCESSED / "dataset_bersih.csv", index=False, encoding="utf-8-sig")
train_df[kolom].to_csv(DATA_PROCESSED / "train.csv", index=False, encoding="utf-8-sig")
test_df[kolom].to_csv(DATA_PROCESSED / "test.csv", index=False, encoding="utf-8-sig")
print("\nTersimpan ke:", DATA_PROCESSED)
for f in ["dataset_bersih.csv", "train.csv", "test.csv"]:
    p = DATA_PROCESSED / f
    print(f" - {f} ({p.stat().st_size} byte)")

Latih: (3614, 14) | Uji: (904, 14)

Proporsi label latih:
label
1    0.5634
0    0.4366
Name: proportion, dtype: float64
Proporsi label uji:
label
1    0.5631
0    0.4369
Name: proportion, dtype: float64

Tersimpan ke: C:\skripsi\data\processed
 - dataset_bersih.csv (816872 byte)
 - train.csv (655269 byte)
 - test.csv (161657 byte)


## Catatan

- Data mentah tetap utuh di `data/raw/`; hasil olahan disimpan terpisah di `data/processed/`.
- Ulasan netral (bintang 3) tidak disertakan pada klasifikasi biner ini.
- Kata negasi dipertahankan karena berpengaruh pada polaritas sentimen.
- TF-IDF dan pemodelan Random Forest dilakukan pada `04_model_klasifikasi.ipynb` dengan *fit* hanya pada data latih.